# Day 022 Project Solution — REST API Client

A `PostsAPI` client wrapping JSONPlaceholder with auth headers, pagination, safe error handling, and AI analysis.

In [ ]:
import requests
import json
import ollama


def get_json(url: str, params: dict | None = None, headers: dict | None = None):
    response = requests.get(url, params=params, headers=headers, timeout=10)
    response.raise_for_status()
    return response.json()


def build_headers(api_key: str, extra: dict | None = None) -> dict:
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    if extra:
        headers.update(extra)
    return headers


def paginate_collect(url: str, page_size: int = 10, max_pages: int = 3) -> list[dict]:
    all_results = []
    for page in range(1, max_pages + 1):
        params = {"_page": page, "_limit": page_size}
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        if not data:
            break
        all_results.extend(data)
    return all_results


def safe_get(url: str, headers: dict | None = None, timeout: int = 10) -> dict:
    try:
        r = requests.get(url, headers=headers, timeout=timeout)
        r.raise_for_status()
        return {"status": r.status_code, "data": r.json(), "error": None}
    except requests.exceptions.HTTPError as e:
        return {"status": e.response.status_code, "data": None, "error": str(e)}
    except Exception as e:
        return {"status": None, "data": None, "error": str(e)}


def ai_analyze_results(records: list[dict], question: str, model: str = "llama3.2") -> str:
    summary = json.dumps(records[:5], indent=2)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are an API data analyst. Answer questions about the provided JSON data concisely.",
            },
            {
                "role": "user",
                "content": f"Data (first 5 records):\n{summary}\n\nQuestion: {question}",
            },
        ],
    )
    return response["message"]["content"]


class PostsAPI:
    BASE = "https://jsonplaceholder.typicode.com"

    def __init__(self, api_key: str = "demo"):
        self.session = requests.Session()
        self.session.headers.update(build_headers(api_key))

    def get_posts(self, user_id: int | None = None, limit: int = 10) -> list[dict]:
        params = {"_limit": limit}
        if user_id is not None:
            params["userId"] = user_id
        r = self.session.get(f"{self.BASE}/posts", params=params, timeout=10)
        r.raise_for_status()
        return r.json()

    def get_pages(self, page_size: int = 5, max_pages: int = 3) -> list[dict]:
        return paginate_collect(f"{self.BASE}/posts", page_size=page_size, max_pages=max_pages)

    def ai_summary(self, question: str, limit: int = 5) -> str:
        posts = self.get_posts(limit=limit)
        return ai_analyze_results(posts, question)

    def safe_fetch(self, endpoint: str) -> dict:
        return safe_get(f"{self.BASE}/{endpoint}")

## Action 1 — Fetch 5 Posts and Print Titles

In [ ]:
client = PostsAPI(api_key='demo')
posts = client.get_posts(limit=5)
print(f'Fetched {len(posts)} posts:')
for p in posts:
    print(f"  [{p['id']}] {p['title']}")

## Action 2 — Filter by User and Paginate

In [ ]:
user_posts = client.get_posts(user_id=1, limit=3)
print(f'\nUser 1 posts (limited to 3): {len(user_posts)}')
for p in user_posts:
    print(f"  userId={p['userId']} | {p['title'][:50]}")

pages = client.get_pages(page_size=5, max_pages=2)
print(f'\nPaginated (2 pages x 5): {len(pages)} total items')

## Action 3 — AI Summary

In [ ]:
summary = client.ai_summary(
    'What topics or themes appear across these posts? Give a one-paragraph overview.',
    limit=5,
)
print('\nAI Summary:')
print(summary)

## Safe Fetch Demo

In [ ]:
good = client.safe_fetch('posts/1')
bad  = client.safe_fetch('nonexistent/endpoint')
print(f'\nSafe fetch /posts/1  -> status={good["status"]}, error={good["error"]}')
print(f'Safe fetch /nonexist -> status={bad["status"]},  error={str(bad["error"])[:40]!r}')
print('\nDemo complete!')